# Data preparation

## Data preparation for BERT fine-tuning

In [1]:
dataset_movie = [
    ("Phim này đúng là một tuyệt tác!", "Positive"),
    ("Xem phí tiền thật sự, không đáng xem!", "Negative"),
    ("Diễn xuất đỉnh cao, cốt truyện cuốn hút từ đầu đến cuối.", "Positive"),
    ("Nội dung nhạt nhẽo và dễ đoán quá.", "Negative"),
    ("Hình ảnh và âm thanh đều rất tuyệt vời!", "Positive"),
]
dataset_product = [
    ("Giao hàng siêu nhanh, đóng gói rất cẩn thận.", "Positive"),
    ("Chất lượng sản phẩm quá tệ, khác xa so với mô tả.", "Negative"),
    ("Dùng rất êm và mượt, đáng tiền!", "Positive"),
    ("Hàng bị lỗi, liên hệ shop không thấy hỗ trợ.", "Negative"),
    ("Săn sale được giá tốt, chất lượng ok.", "Positive"),
]
dataset_sentiment = [
    ("Dịch vụ chăm sóc khách hàng ở đây rất nhiệt tình.", "Positive"),
    ("Đồ ăn dở tệ, thái độ nhân viên còn lồi lõm.", "Negative"),
    ("Phòng ốc tạm ổn, không có gì quá nổi bật.", "Neutral"),
    ("Không gian quán thoáng mát, đồ uống ngon.", "Positive"),
    ("Giá cả bình thường, chất lượng ở mức chấp nhận được.", "Neutral"),
]

In [2]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

/home/lai/Documents/domain-specific-sml/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
max_length = 128
formated_data = [(f"[CLS] {text} [SEP]", label) for text, label in dataset_movie]
formated_data

[('[CLS] Phim này đúng là một tuyệt tác! [SEP]', 'Positive'),
 ('[CLS] Xem phí tiền thật sự, không đáng xem! [SEP]', 'Negative'),
 ('[CLS] Diễn xuất đỉnh cao, cốt truyện cuốn hút từ đầu đến cuối. [SEP]',
  'Positive'),
 ('[CLS] Nội dung nhạt nhẽo và dễ đoán quá. [SEP]', 'Negative'),
 ('[CLS] Hình ảnh và âm thanh đều rất tuyệt vời! [SEP]', 'Positive')]

In [4]:
tokenized_data = tokenizer(
    formated_data,
    padding=True,
    truncation=True,
    max_length=max_length,
    return_tensors="pt"
)

In [5]:
tokenized_data[0]

Encoding(num_tokens=30, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])

In [6]:
# tokenized_data[0].offsets

 ### 1. ids

  Danh sách các token ID (số nguyên) — mỗi token trong câu được ánh xạ sang một số nguyên tương ứng trong bảng từ vựng (vocabulary) của mô hình. Đây là đầu vào chính mà mô hình BERT
  sử dụng.

  │ Ví dụ: "hello world" → [101, 7592, 2088, 102] (trong đó 101 = [CLS], 102 = [SEP])

  ### 2. type_ids

  Danh sách các segment ID (hay còn gọi là token_type_ids) — dùng để phân biệt câu A và câu B trong các tác vụ gồm cặp câu (sentence-pair tasks).

  • 0 = thuộc câu thứ nhất (Segment A)
  • 1 = thuộc câu thứ hai (Segment B)

  │ Ví dụ: [CLS] Tôi thích AI [SEP] AI rất hay [SEP] → [0, 0, 0, 0, 0, 1, 1, 1, 1]

  ### 3. tokens

  Danh sách các token dạng chuỗi — kết quả sau khi tách từ (tokenization). Giúp bạn quan sát trực tiếp mô hình đã tách câu thành những phần nào.

  │ Ví dụ: "unbelievable" → ["un", "##believ", "##able"] (WordPiece tokenization)

  ### 4. offsets

  Danh sách các cặp (start, end) — chỉ vị trí bắt đầu và kết thúc của mỗi token trong chuỗi gốc (original string). Rất hữu ích khi cần ánh xạ ngược từ token về vị trí trong văn bản
  ban đầu (ví dụ: trong NER).

  │ Ví dụ: "Hello" → token "hello" có offset (0, 5)

  • Các special token ([CLS], [SEP], [PAD]) thường có offset là (0, 0).

  ### 5. attention_mask

  Danh sách các giá trị 0 hoặc 1 — cho mô hình biết token nào là thật và token nào là padding.

  • 1 = token thật, mô hình cần chú ý (attend) đến
  • 0 = token padding, mô hình sẽ bỏ qua

  │ Ví dụ: [1, 1, 1, 1, 0, 0] → 4 token thật, 2 token padding

  ### 6. special_tokens_mask

  Danh sách các giá trị 0 hoặc 1 — đánh dấu token nào là special token (token đặc biệt do tokenizer tự thêm vào).

  • 1 = special token ([CLS], [SEP], [PAD])
  • 0 = token bình thường từ văn bản đầu vào

  │ Ví dụ: [CLS] hello world [SEP] → [1, 0, 0, 1]

  ### 7. overflowing

  Danh sách các Encoding phụ — chứa các token bị tràn ra khi câu dài hơn max_length. Khi bật truncation=True, phần vượt quá max_length sẽ được lưu ở đây (nếu tokenizer được cấu hình
  return_overflowing_tokens=True).

  • Nếu câu không bị tràn → danh sách rỗng []
  • Nếu câu bị tràn → chứa 1 hoặc nhiều Encoding object cho phần còn lại
  ──────
  ### Tóm tắt nhanh

   Thuộc tính                                                 | Kiểu dữ liệu                                               | Ý nghĩa
  ------------------------------------------------------------|------------------------------------------------------------|-----------------------------------------------------------
   ids                                                        | List[int]                                                  | ID số nguyên của từng token
   type_ids                                                   | List[int]                                                  | Phân biệt câu A (0) / câu B (1)
   tokens                                                     | List[str]                                                  | Token dạng chuỗi
   offsets                                                    | List[(int, int)]                                           | Vị trí (start, end) trong chuỗi gốc
   attention_mask                                             | List[int]                                                  | Đánh dấu token thật (1) / padding (0)
   special_tokens_mask                                        | List[int]                                                  | Đánh dấu special token (1) / token thường (0)
   overflowing                                                | List[Encoding]                                             | Các token bị tràn khi vượt max_length

──────────────────────────────────────────────────────────────

In [7]:
import torch
from sklearn.preprocessing import LabelEncoder

In [8]:
input_ids = tokenized_data['input_ids']
attention_mask = tokenized_data['attention_mask']
labels = torch.tensor(LabelEncoder().fit_transform([label for _, label in dataset_movie]))

In [9]:
labels

tensor([1, 0, 1, 0, 1])

In [10]:
from sklearn.model_selection import train_test_split

train_inputs, val_inputs, train_labels, val_labels, train_mask, val_mask = train_test_split(
    input_ids, labels, attention_mask, random_state=42, test_size=0.1
)

In [11]:
# Create Dataset object

from torch.utils.data import Dataset

class CustomDataset(Dataset):
    def __init__(self, input_ids, attention_mask, labels):
        self.input_ids = input_ids
        self.attention_mask = attention_mask
        self.labels = labels

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return {
            'input_ids': self.input_ids[idx],
            'attention_mask': self.attention_mask[idx],
            'labels': self.labels[idx]
        }


In [12]:
# Create Dataloader

from torch.utils.data import DataLoader

batch_size = 4
train_dataset = CustomDataset(train_inputs, train_mask, train_labels)
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

val_dataset = CustomDataset(val_inputs, val_mask, val_labels)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=True)

In [13]:
# Load model from Huggingface hub
from transformers import BertForSequenceClassification

model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=len(set(labels)))

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7803.03it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoi

## Data preparation for GPT Fine-tuning

In [15]:
from dataset_gpt_vi import dataset_gpt as dataset
dataset

[('Ngày xửa ngày xưa, ở một ngôi làng nhỏ,',
  'có một cô gái mồ côi sống với bà ngoại trong căn nhà tranh rách nát.'),
 ('Trong khu rừng sâu thẳm,',
  'có một con rồng già canh giữ kho báu từ nghìn năm trước.'),
 ('Vị vua già nhìn ra ngoài cửa sổ và thở dài,',
  'vì ba hoàng tử đều không ai chịu kế thừa ngai vàng.'),
 ('Chú bé chăn trâu ngồi trên lưng trâu,',
  'thổi sáo một bài ca vui vẻ, tiếng sáo vang khắp cánh đồng.'),
 ('Trí tuệ nhân tạo đang thay đổi thế giới,',
  'từ y tế, giáo dục cho đến sản xuất công nghiệp đều được tự động hóa.'),
 ('Để huấn luyện một mô hình ngôn ngữ lớn,',
  'bạn cần một lượng dữ liệu khổng lồ và tài nguyên tính toán mạnh mẽ.'),
 ('Học sâu (Deep Learning) là một nhánh của',
  'học máy, sử dụng mạng nơ-ron nhiều tầng để học các biểu diễn phức tạp từ dữ liệu.'),
 ('Khi fine-tune một mô hình GPT,',
  'bước đầu tiên là chuẩn bị dữ liệu chất lượng cao phù hợp với miền ứng dụng.'),
 ('Phở là món ăn truyền thống nổi tiếng nhất của Việt Nam,',
  'với nước dùng đậ

In [35]:
from transformers import GPT2Tokenizer

tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token
formatted_data = [(f"[CLS] {context} [SEP] {target} [SEP]",) for context, target in dataset]
formatted_data

[('[CLS] Ngày xửa ngày xưa, ở một ngôi làng nhỏ, [SEP] có một cô gái mồ côi sống với bà ngoại trong căn nhà tranh rách nát. [SEP]',),
 ('[CLS] Trong khu rừng sâu thẳm, [SEP] có một con rồng già canh giữ kho báu từ nghìn năm trước. [SEP]',),
 ('[CLS] Vị vua già nhìn ra ngoài cửa sổ và thở dài, [SEP] vì ba hoàng tử đều không ai chịu kế thừa ngai vàng. [SEP]',),
 ('[CLS] Chú bé chăn trâu ngồi trên lưng trâu, [SEP] thổi sáo một bài ca vui vẻ, tiếng sáo vang khắp cánh đồng. [SEP]',),
 ('[CLS] Trí tuệ nhân tạo đang thay đổi thế giới, [SEP] từ y tế, giáo dục cho đến sản xuất công nghiệp đều được tự động hóa. [SEP]',),
 ('[CLS] Để huấn luyện một mô hình ngôn ngữ lớn, [SEP] bạn cần một lượng dữ liệu khổng lồ và tài nguyên tính toán mạnh mẽ. [SEP]',),
 ('[CLS] Học sâu (Deep Learning) là một nhánh của [SEP] học máy, sử dụng mạng nơ-ron nhiều tầng để học các biểu diễn phức tạp từ dữ liệu. [SEP]',),
 ('[CLS] Khi fine-tune một mô hình GPT, [SEP] bước đầu tiên là chuẩn bị dữ liệu chất lượng cao phù h

In [36]:
numerical_data = [tokenizer.encode(example[0], add_special_tokens=True) for example in formatted_data]

In [37]:
# numerical_data

In [38]:
import torch

max_length = max(len(seq) for seq in numerical_data)
max_length

151

In [39]:
padded_data = [seq + [tokenizer.pad_token_id]*(max_length - len(seq)) for seq in numerical_data]
input_ids = torch.tensor(padded_data)

In [40]:
from torch.utils.data import Dataset

class CustomDataset(Dataset):
    def __init__(self, input_ids):
        self.input_ids = input_ids

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return {
            'input_ids': self.input_ids[idx]
        }

from torch.utils.data import DataLoader

batch_size = 4
custom_dataset = CustomDataset(input_ids)
dataloader = DataLoader(custom_dataset, batch_size=batch_size, shuffle=True)

In [41]:
from transformers import GPT2LMHeadModel

model = GPT2LMHeadModel.from_pretrained('gpt2')

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 6393.03it/s]
